# KrickBot Llama 3 Fine-Tuning (Google Colab)

This notebook uses the `unsloth` library to fine-tune Llama 3 8B in 4-bit quantization, which allows it to fit easily on a free Google Colab T4 GPU.

### Instructions
1. In Colab, go to **Runtime > Change runtime type** and select **T4 GPU**.
2. Make sure `refined_dataset.jsonl` is in your Google Drive at `MyDrive/krickbot/`.
3. Run all cells.
4. Download the resulting `.gguf` file to use locally.

### Checkpoint & Resume
- Checkpoints are saved every **250 steps** directly to Google Drive.
- If training crashes or Colab disconnects, just **re-run all cells** — training will automatically resume from the last checkpoint.
- To start fresh, delete the `outputs/` folder in your Google Drive before running.

In [ ]:
# 1. Install Dependencies
!pip install unsloth
!pip install "transformers>=4.46.0,<4.52.0"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

In [ ]:
# 2. Suppress noisy warnings & fix HF download issues
# MUST run before any other imports
!rm -rf ~/.cache/huggingface/hub  # Clear any corrupted downloaded files
import os
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["HF_HUB_DISABLE_XET"] = "1"

import logging
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("transformers.utils.import_utils").setLevel(logging.ERROR)

In [ ]:
# 3. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Verify dataset exists
DATASET_PATH = "/content/drive/MyDrive/krickbot/refined_dataset.jsonl"
assert os.path.exists(DATASET_PATH), f"Dataset not found at {DATASET_PATH}"
print(f"\u2705 Dataset found: {DATASET_PATH}")

In [ ]:
# 4. Load Model & Tokenizer
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True

# Load base model (Llama-3 8B Instruct)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Attach LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

In [ ]:
# 5. Prepare Dataset
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3",
    mapping = {"role" : "role", "content" : "content", "user" : "user", "assistant" : "model"},
)

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts, }

from datasets import load_dataset

print("Loading refined_dataset.jsonl from Google Drive...")
dataset = load_dataset("json", data_files=DATASET_PATH, split="train")
dataset = dataset.map(formatting_prompts_func, batched = True,)
print(f"\u2705 Dataset loaded: {len(dataset)} examples")

In [ ]:
# 6. Train the Model (with checkpointing & auto-resume)
import glob
from trl import SFTTrainer, SFTConfig
import os

# --- Checkpoint resume detection ---
# Save checkpoints DIRECTLY to Google Drive so they survive timeouts
output_dir = "/content/drive/MyDrive/krickbot/outputs"
resume_from = None

if os.path.isdir(output_dir):
    checkpoints = sorted(
        glob.glob(os.path.join(output_dir, "checkpoint-*")),
        key=lambda x: int(x.split("-")[-1])
    )
    if checkpoints:
        resume_from = checkpoints[-1]
        print(f"\n\U0001f504 Resuming training from: {resume_from}")
        print(f"   (Found {len(checkpoints)} checkpoint(s))\n")
    else:
        print("\n\U0001f195 No checkpoints found. Starting fresh training.\n")
else:
    print("\n\U0001f195 No output directory found. Starting fresh training.\n")

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_num_proc = 2,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 1,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = output_dir,

        # --- Checkpoint settings ---
        save_strategy = "steps",
        save_steps = 250,
        save_total_limit = 3,

        # --- SFT-specific settings ---
        dataset_text_field = "text",
        max_seq_length = max_seq_length,
        packing = False,
    ),
)

# --- Train (with auto-resume) ---
trainer_stats = trainer.train(resume_from_checkpoint=resume_from)

print("\n\u2705 Training complete!")
print(f"   Total steps: {trainer_stats.global_step}")
print(f"   Final loss:  {trainer_stats.training_loss:.4f}")

In [ ]:
# 7. Export Model to GGUF
model.save_pretrained_gguf("/content/drive/MyDrive/krickbot/krickbot_model", tokenizer, quantization_method = "q8_0")
print("\u2705 Model successfully exported to GGUF in your Google Drive!")
print("You can now download the krickbot_model-unsloth.Q8_0.gguf file from your Drive.")